# 04 · Synthesis, recommendations and limitations

**Author:** Yike SHI

**Question 4 of the brief.** This notebook gathers the headline numbers produced by notebooks
01–03 (read from `outputs/results/`, so every figure quoted here is traceable), discusses where the
indicators agree and where they conflict, and draws recommendations and limitations.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import config as C

R = lambda name, **kw: pd.read_csv(C.RES_DIR / f"{name}.csv", **kw)
eff = R("table03_effort_by_year", index_col=0)
ind = R("table09_annual_indicators", index_col=0)
lad = R("table10_model_ladder", index_col=0)
adj = R("table12_adjusted_indices", header=[0, 1], index_col=0)
rob = R("table13_robustness", index_col=0)
oth = R("table14_other_indicator_trends", index_col=0)
spr = R("table17_species_rates", index_col=0)
drv = R("table18_exotic_drivers", index_col=0)

def fmt(e, l, h, d=1, unit=""):
    return f"{e:+.{d}f}{unit} [{l:+.{d}f}, {h:+.{d}f}]"

m4a = lad.loc["M4a + observer FE"]; m3 = lad.loc["M3 + conditions"]
headline = pd.DataFrame([
    ("Effort", "visits per year", f"{eff['visits'].iloc[0]} (2014) → {eff['visits'].iloc[-1]} (2025)", "census, no CI"),
    ("Abundance index", "raw base-100, 2022 and 2025",
     f"{ind.loc[2022, 'abundance_base100']:.0f} [{ind.loc[2022, 'base100_low']:.0f}, {ind.loc[2022, 'base100_high']:.0f}]; "
     f"{ind.loc[2025, 'abundance_base100']:.0f} [{ind.loc[2025, 'base100_low']:.0f}, {ind.loc[2025, 'base100_high']:.0f}]", "cluster bootstrap"),
    ("Abundance index", "adjusted (sites, conditions, observers), 2025",
     f"{adj[('adjusted M4a (+ observers)', 'index100')].loc[2025]:.0f} "
     f"[{adj[('adjusted M4a (+ observers)', 'low')].loc[2025]:.0f}, {adj[('adjusted M4a (+ observers)', 'high')].loc[2025]:.0f}]", "cluster bootstrap"),
    ("Abundance trend", "M3 (no observers) → M4a (observer FE), % per year",
     f"{fmt(m3['pct_per_year'], m3['ci_low'], m3['ci_high'], 2)} → {fmt(m4a['pct_per_year'], m4a['ci_low'], m4a['ci_high'], 2)}", "cluster bootstrap"),
    ("Abundance trend", "M4a on core panel / within 50 m / without 2014",
     " / ".join(fmt(*rob.loc[k], 2) for k in ["core panel only (40 transects, every year)", "birds within 50 m only", "excluding 2014 (ramp-up year)"]), "cluster bootstrap"),
    ("Local richness", "species per visit, slope with observer FE",
     fmt(*oth.loc["species per visit, M4a", ["slope", "ci_low", "ci_high"]], 3, " sp/yr"), "cluster bootstrap"),
    ("Island richness", "equal-effort (60 visits), slope",
     fmt(*oth.loc["rarefied island richness (m = 60) [jackknife]", ["slope", "ci_low", "ci_high"]], 2, " sp/yr"), "jackknife"),
    ("Composition", "exotic share, slope (core panel)",
     fmt(*oth.loc["exotic share, core panel", ["slope", "ci_low", "ci_high"]], 2, " pts/yr"), "cluster bootstrap"),
    ("Species", "classes (34 species, estimator B)",
     ", ".join(f"{k}: {v}" for k, v in spr["class_B"].value_counts().items() if not k.startswith("not")), "cluster bootstrap"),
], columns=["indicator", "quantity", "estimate [95% CI]", "uncertainty method"])
headline.to_csv(C.RES_DIR / "table19_headline_results.csv", index=False)
headline

,indicator,quantity,estimate [95% CI],uncertainty method
0,Effort,visits per year,78 (2014) → 129 (2025),"census, no CI"
1,Abundance index,"raw base-100, 2022 and 2025","134 [121, 150]; 112 [101, 124]",cluster bootstrap
2,Abundance index,"adjusted (sites, conditions, observers), 2025","113 [104, 122]",cluster bootstrap
3,Abundance trend,"M3 (no observers) → M4a (observer FE), % per year","+1.09 [+0.49, +1.66] → +0.74 [+0.24, +1.25]",cluster bootstrap
4,Abundance trend,M4a on core panel / within 50 m / without 2014,"+0.49 [-0.07, +1.02] / +0.28 [-0.40, +0.91] / ...",cluster bootstrap
5,Local richness,"species per visit, slope with observer FE","+0.034 sp/yr [-0.019, +0.085]",cluster bootstrap
6,Island richness,"equal-effort (60 visits), slope","+0.13 sp/yr [-0.33, +0.59]",jackknife
7,Composition,"exotic share, slope (core panel)","+0.45 pts/yr [+0.16, +0.74]",cluster bootstrap
8,Species,"classes (34 species, estimator B)","stable: 11, uncertain: 10, moderate increase: ...",cluster bootstrap


## 1. Main insights

1. **The monitoring programme is informative but its effort is not constant.** Effort grew until
   2018, the transect panel changed, and the observer pool contracted (2020–2023) then expanded
   (2024). Raw totals are not comparable across years; standardised indices and models are.
2. **Bird numbers did not decline over 2014–2025.** The standardised community index rose to a
   maximum in 2020–2022 and returned to about +12 % above 2014 by 2025. After adjusting for sites,
   survey conditions and observers, the long-run rate is small and positive (≈ +0.7 % per year)
   but not robust: restricted to the core panel, to birds within 50 m, or without the 2014
   ramp-up year, its interval includes zero.
3. **Diversity is stable, composition is not.** Species richness per visit and island-wide
   richness at equal effort show no clear change, while the share of individuals belonging to
   exotic species rose from ≈ 10 % to ≈ 15 %, driven by four colonisers (Eurasian Collared-Dove,
   Common Waxbill, Shiny Cowbird, Cattle Egret).
4. **Endemic species carry most of the negative signals.** All three endemic hummingbirds, the
   Scaly-breasted Thrasher (Moqueur grivotte) and both Martinique endemics have negative rate
   estimates; the Martinique (White-breasted) Thrasher's index fell by about two thirds on its
   four transects.
5. **Who counts matters.** About a third of the naive abundance trend, most of the naive increase
   in species per visit, and the trend class of 15 of 34 species depend on accounting for
   observers. A simulation with no true trend reproduces an observer-induced bias of the same size.

## 2. Converging evidence and contradictions

**Converging**
* Every adjustment for effort and observers *reduces* an apparent increase (abundance rate,
  species per visit, several species classes, raw vs rarefied richness): the same mechanism —
  more effort, a changing observer mix and a drift towards detecting more distant birds —
  inflates naive trends upward.
* The community-level rise of exotics (notebook 02) and the species-level increases of the
  colonisers (notebook 03; Collared-Dove, Waxbill, Shiny Cowbird) are two views of the same change; likewise, the slight decline of
  the endemic share matches the negative endemic species trends.

**Contradictions and tensions**
* *Abundance up, endemics down.* The modest increase in total abundance is carried by doves,
  exotic colonisers and a few common natives, not by endemics: a stable or rising total can hide
  the erosion of the species the island is responsible for. Total abundance alone would be a
  misleading biodiversity indicator.
* *Linear vs non-linear.* A single rate summarises the abundance series poorly (rise, plateau,
  decline); model comparison favours a flexible year effect. Rates are useful for ranking species,
  less for describing the community's trajectory.
* *Model criteria disagree.* AIC and cross-validated error favour observer fixed effects; BIC
  prefers the leaner experience / first-year models. Since the year estimate depends on this
  choice, both are reported.
* *Estimators disagree for some species* (annual log-linear vs visit-level model), most strongly
  for flocking and clustered species.

## 3. Recommendations

**Data collection**
1. *Structured digital entry form* with built-in checks (one date and one passage per visit,
   points 1–10, species from a fixed list, no formula totals). It would have prevented the 25
   double-entered visits, drag-filled dates and broken totals found in notebook 01.
2. *Record effort explicitly*: a visit identifier, observer codes, start and end time of every
   point, and an explicit "nothing detected" line, so that effort never has to be reconstructed.
3. *Manage the observer design*: avoid one observer covering more than half of the visits;
   schedule overlapping visits (two observers on the same transect, or swaps between years) to
   estimate observer effects; keep training/mentoring in a dedicated field from the start (it only
   appears in the remarks from 2021).
4. *Separate detection from abundance*: distance bands are already recorded — use them for
   distance sampling; recording the minute of first detection would allow removal models; the two
   passages per year already support occupancy/N-mixture models.
5. *Codebook for weather* (visibility is coded inconsistently between observers) and keep both
   passages every year (2020 lost ten first passages).

**Biodiversity management and follow-up questions**
6. *Targeted monitoring* of the Martinique Thrasher on the Caravelle peninsula (dedicated counts
   and predator control monitoring) and nest monitoring of the Martinique Oriole to measure Shiny
   Cowbird parasitism rates.
7. *Surveillance of spreading exotics* (Eurasian Collared-Dove, Common Waxbill) and of their
   habitats, to anticipate impacts on native doves and finches.
8. *Hummingbirds*: link counts to flowering resources and extreme events to test the hurricane
   and drought hypotheses.

## 4. Limitations

* **Detection.** All indices measure *detected* birds. Observer and site effects are modelled,
  but species-, habitat- and distance-specific detectability are not; the growing share of distant
  detections shows the detection process changed.
* **Inference target.** Transects were not drawn at random from the island; the cluster bootstrap
  treats them as exchangeable units, so conclusions apply to the *monitored sites* and their
  habitats, not automatically to all of Martinique.
* **Short series.** Twelve years, with a ramp-up first year, give low power to separate trends
  from fluctuations; rates are sensitive to the endpoints.
* **Model assumptions.** Constant variance on the log scale was checked (notebook 02 diagnostics,
  mild heavy tails); the species-level model on raw counts violates it and relies on the bootstrap
  for valid intervals. Observer experience is too collinear with time to be estimated precisely.
* **Methodological sensitivity.** Conclusions that depend on choices are reported as such:
  observer specification, linear vs flexible shape, core panel, distance restriction, estimator A
  vs B. Cleaning removed about 2 % of rows (double entries, conflicting fragments); every rule and its row count is in the cleaning log.